# 03 — Transformation Bronze vers Silver | Panier Vert

**Objectif :** lire les fichiers Bronze sans les modifier, appliquer des nettoyages prudents et enregistrer les résultats au format Parquet dans `data/silver/`.

**Principes :**
- Bronze reste immuable.
- Pas de suppression automatique de lignes uniquement parce qu'une valeur est manquante.
- Les identifiants sont conservés autant que possible.
- Les transformations et contrôles sont tracés dans un petit rapport.
- Exécuter les cellules dans l'ordre depuis la racine du projet.

In [17]:
from pathlib import Path
import os
import sys
import io
import re
import json
import tempfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Retrouver la racine du projet
cwd = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "src" / "panier_vert" / "storage" / "rustfs.py").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Racine du projet introuvable. "
        "Vérifie que le notebook est dans le projet Panier Vert."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from panier_vert.storage.rustfs import s3, RUSTFS_BUCKET

print("Projet :", PROJECT_ROOT)
print("Bucket RustFS :", RUSTFS_BUCKET)
print("Connexion RustFS prête.")

Projet : c:\Users\akono\projet_mspr_panier_vert
Bucket RustFS : panier-vert
Connexion RustFS prête.


In [19]:
def list_s3_objects(prefix):
    paginator = s3.get_paginator("list_objects_v2")

    keys = []

    for page in paginator.paginate(
        Bucket=RUSTFS_BUCKET,
        Prefix=prefix
    ):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])

    return keys


bronze_files = sorted([
    key for key in list_s3_objects("bronze/")
    if key.lower().endswith(".parquet")
    and "_smoke/" not in key
    and "_runs/" not in key
])

inventory = []

for key in bronze_files:
    metadata = s3.head_object(
        Bucket=RUSTFS_BUCKET,
        Key=key
    )

    inventory.append({
        "fichier_bronze": key,
        "taille_mo": round(
            metadata["ContentLength"] / (1024 ** 2), 2
        )
    })

df_inventory = pd.DataFrame(inventory)

display(df_inventory)

print("Nombre de fichiers Bronze :", len(bronze_files))

assert len(bronze_files) == 9, (
    f"9 fichiers attendus, {len(bronze_files)} trouvés. "
    "Vérifie le transfert Bronze avant de continuer."
)

,fichier_bronze,taille_mo
0,bronze/instacart/aisles.parquet,0.00
1,bronze/instacart/departments.parquet,0.00
2,bronze/instacart/order_products__prior.parquet,125.81
3,bronze/instacart/order_products__train.parquet,5.37
4,bronze/instacart/orders.parquet,31.60
5,bronze/instacart/products.parquet,1.45
6,bronze/openfoodfacts/products_20261117.parquet,0.21
7,bronze/rappelconso/rappelconso.parquet,4.36
8,bronze/web_source/articles_20261117.parquet,0.05


Nombre de fichiers Bronze : 9


In [20]:
def clean_scalar(value):
    """Nettoyer une valeur textuelle simple."""
    if value is None or value is pd.NA:
        return pd.NA

    if isinstance(value, (list, tuple, dict, set, np.ndarray)):
        return value

    try:
        if pd.isna(value):
            return pd.NA
    except (TypeError, ValueError):
        pass

    value = str(value)
    value = value.replace("\u00a0", " ")
    value = value.replace("\u200b", "")
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA


def clean_list(value):
    """Nettoyer les éléments d'une liste sans perdre sa structure."""
    if value is None or value is pd.NA:
        return None

    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        result = []

        for item in value:
            item = clean_scalar(item)

            if item is pd.NA:
                continue

            if str(item).lower() in {
                "none", "null", "nan", "en:none", ""
            }:
                continue

            result.append(str(item))

        return result if result else None

    return clean_scalar(value)


def clean_dataframe_text(df, excluded=None):
    """Nettoyer les colonnes texte en préservant les listes."""
    excluded = set(excluded or [])

    for col in df.columns:

        if col in excluded:
            continue

        if (
            pd.api.types.is_object_dtype(df[col])
            or pd.api.types.is_string_dtype(df[col])
        ):

            non_null = df[col].dropna()

            has_nested_values = any(
                isinstance(v, (list, tuple, set, dict, np.ndarray))
                for v in non_null
            )

            if has_nested_values:
                df[col] = df[col].map(clean_list)
            else:
                df[col] = (
                    df[col]
                    .map(clean_scalar)
                    .astype("string")
                )

    return df


def convert_integer_columns(df, columns):
    """Convertir les colonnes numériques entières présentes."""
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            ).astype("Int64")

    return df


def clean_dates(df, tokens):
    """
    Ajouter des colonnes de dates analysées.
    Les colonnes originales sont conservées.
    """
    for col in list(df.columns):

        if any(token in col.lower() for token in tokens):

            parsed = pd.to_datetime(
                df[col],
                errors="coerce",
                format="mixed",
                dayfirst=True,
                utc=True
            )

            parsed_col = f"{col}_parsed"

            if parsed_col not in df.columns:
                df[parsed_col] = parsed

    return df

In [22]:
INTEGER_COLUMNS = [
    "order_id",
    "user_id",
    "product_id",
    "aisle_id",
    "department_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "add_to_cart_order",
    "reordered"
]


def clean_instacart(df):

    df.columns = [
        str(col).strip().lower()
        for col in df.columns
    ]

    df = clean_dataframe_text(df)

    df = convert_integer_columns(
        df,
        INTEGER_COLUMNS
    )

    if "days_since_prior_order" in df.columns:

        df["days_since_prior_order"] = pd.to_numeric(
            df["days_since_prior_order"],
            errors="coerce"
        )

        df.loc[
            df["days_since_prior_order"] < 0,
            "days_since_prior_order"
        ] = np.nan

    anomalies = {}

    # Jour de la semaine : 0 à 6
    if "order_dow" in df.columns:

        mask = (
            df["order_dow"].notna()
            & ~df["order_dow"].between(0, 6)
        )

        anomalies["order_dow_hors_plage"] = int(mask.sum())

        df.loc[mask, "order_dow"] = pd.NA

    # Heure : 0 à 23
    if "order_hour_of_day" in df.columns:

        mask = (
            df["order_hour_of_day"].notna()
            & ~df["order_hour_of_day"].between(0, 23)
        )

        anomalies["order_hour_hors_plage"] = int(mask.sum())

        df.loc[mask, "order_hour_of_day"] = pd.NA

    # Reordered : 0 ou 1
    if "reordered" in df.columns:

        mask = (
            df["reordered"].notna()
            & ~df["reordered"].isin([0, 1])
        )

        anomalies["reordered_hors_0_1"] = int(mask.sum())

        df.loc[mask, "reordered"] = pd.NA

    return df, anomalies

In [23]:
def clean_rappelconso(df):

    df.columns = [
        str(col).strip()
        for col in df.columns
    ]

    identifier_columns = [
        col for col in df.columns
        if col.lower() in {
            "gtin",
            "id",
            "numero_fiche",
            "numero_version"
        }
    ]

    df = clean_dataframe_text(
        df,
        excluded=identifier_columns
    )

    for col in identifier_columns:
        df[col] = (
            df[col]
            .map(clean_scalar)
            .astype("string")
        )

    df = clean_dates(
        df,
        ["date", "publication"]
    )

    anomalies = {}

    for col in df.columns:

        if col.lower() == "gtin":

            anomalies["gtin_manquants"] = int(
                df[col].isna().sum()
            )

            anomalies["gtin_doublons"] = int(
                df[col].notna().sum()
                - df.loc[df[col].notna(), col].nunique()
            )

    return df, anomalies

In [24]:
BATCH_SIZE = 250_000

def get_source(object_key):

    parts = object_key.split("/")

    if len(parts) < 3:
        raise ValueError(f"Chemin Bronze inattendu : {object_key}")

    source = parts[1]

    if source == "scraping":
        source = "web_source"

    return source


def get_silver_key(object_key):

    parts = object_key.split("/")

    source = get_source(object_key)
    filename = "/".join(parts[2:])

    return f"silver/{source}/{filename}"


def clean_batch(df, source):

    if source == "instacart":
        return clean_instacart(df)

    elif source == "openfoodfacts":
        return clean_openfoodfacts(df)

    elif source == "rappelconso":
        return clean_rappelconso(df)

    elif source == "web_source":
        return clean_web_source(df)

    else:
        raise ValueError(
            f"Source non reconnue : {source}"
        )


def process_bronze_to_silver(object_key):

    source = get_source(object_key)
    destination = get_silver_key(object_key)

    print("\nTraitement :", object_key)
    print("Destination :", destination)

    total_input = 0
    total_output = 0
    batch_count = 0
    anomaly_totals = {}

    writer = None

    with tempfile.TemporaryDirectory(
        prefix="panier_silver_"
    ) as temp_dir:

        input_path = Path(temp_dir) / "input.parquet"
        output_path = Path(temp_dir) / "output.parquet"

        # Télécharger le Bronze temporairement
        s3.download_file(
            RUSTFS_BUCKET,
            object_key,
            str(input_path)
        )

        # Lire le Parquet par lots
        with open(input_path, "rb") as input_file:

            parquet_file = pq.ParquetFile(input_file)

            for batch in parquet_file.iter_batches(
                batch_size=BATCH_SIZE
            ):

                df = batch.to_pandas()

                rows_before = len(df)

                cleaned_df, anomalies = clean_batch(
                    df,
                    source
                )

                rows_after = len(cleaned_df)

                total_input += rows_before
                total_output += rows_after
                batch_count += 1

                for name, value in anomalies.items():

                    anomaly_totals[name] = (
                        anomaly_totals.get(name, 0)
                        + int(value)
                    )

                table = pa.Table.from_pandas(
                    cleaned_df,
                    preserve_index=False
                )

                if writer is None:

                    writer = pq.ParquetWriter(
                        output_path,
                        table.schema,
                        compression="snappy"
                    )

                writer.write_table(table)

                print(
                    f"Lot {batch_count} : "
                    f"{rows_before:,} lignes traitées"
                )

                del df, cleaned_df, table, batch

            del parquet_file

        # Fermer le fichier Parquet de sortie
        if writer is not None:
            writer.close()
            writer = None

        # Envoyer le résultat dans RustFS
        if not output_path.exists():
            raise RuntimeError(
                f"Aucun fichier de sortie créé pour {object_key}"
            )

        s3.upload_file(
            str(output_path),
            RUSTFS_BUCKET,
            destination
        )

        # Vérifier l'objet distant
        remote = s3.head_object(
            Bucket=RUSTFS_BUCKET,
            Key=destination
        )

        result = {
            "source": source,
            "bronze": object_key,
            "silver": destination,
            "lignes_entree": total_input,
            "lignes_sortie": total_output,
            "lots_traites": batch_count,
            "taille_silver_octets": remote["ContentLength"],
            "anomalies": json.dumps(
                anomaly_totals,
                ensure_ascii=False
            ),
            "statut": "OK"
        }

        print("Enregistré dans RustFS :", destination)
        print("Lignes :", total_input, "=>", total_output)

        return result

In [25]:
def clean_openfoodfacts(df):
    """
    Nettoyage des données Open Food Facts pour la couche Silver.
    """

    df = df.copy()

    # Nettoyage des valeurs textuelles
    df = clean_dataframe_text(df)

    # Identifiants conservés en string
    for col in ["code", "id", "product_name", "brands"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Conversion des colonnes numériques
    numeric_columns = [
        "nutriscore_score",
        "nova_group",
        "last_modified_t",
        "energy-kcal_100g",
        "fat_100g",
        "saturated-fat_100g",
        "carbohydrates_100g",
        "sugars_100g",
        "proteins_100g",
        "salt_100g",
        "fiber_100g",
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Validation NOVA (1 à 4)
    if "nova_group" in df.columns:
        df.loc[
            ~df["nova_group"].isin([1, 2, 3, 4]),
            "nova_group"
        ] = np.nan

    # Validation Nutri-Score
    if "nutriscore_grade" in df.columns:
        df["nutriscore_grade"] = (
            df["nutriscore_grade"]
            .astype("string")
            .str.lower()
            .str.strip()
        )

        df.loc[
            ~df["nutriscore_grade"].isin(["a", "b", "c", "d", "e"]),
            "nutriscore_grade"
        ] = pd.NA

    # Conversion timestamp
    if "last_modified_t" in df.columns:
        df["last_modified_at"] = pd.to_datetime(
            df["last_modified_t"],
            unit="s",
            errors="coerce",
            utc=True
        )

    # Rapport qualité
    anomalies = {
        "source": "openfoodfacts",
        "rows": len(df),
        "missing_codes": 0,
        "duplicate_codes": 0,
    }

    if "code" in df.columns:
        anomalies["missing_codes"] = int(df["code"].isna().sum())
        anomalies["duplicate_codes"] = int(df["code"].duplicated().sum())

    print(f"Open Food Facts : {len(df):,} lignes nettoyées")

    return df, anomalies

In [26]:
def clean_web_source(df):
    """
    Nettoyage des données issues du scraping pour la couche Silver.
    """

    df = df.copy()

    # Nettoyage des valeurs textuelles
    df = clean_dataframe_text(df)

    # Normalisation des colonnes textuelles
    text_columns = [
        "title",
        "content",
        "description",
        "url",
        "source",
        "author",
        "category",
    ]

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Normalisation des URLs
    if "url" in df.columns:
        df["url"] = df["url"].replace("", pd.NA)

    # Conversion des dates
    date_columns = [
        "date",
        "published_at",
        "publication_date",
        "scraped_at",
        "created_at",
    ]

    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce",
                utc=True
            )

    # Détection des titres manquants
    missing_titles = 0

    if "title" in df.columns:
        missing_titles = int(
            df["title"].isna().sum()
            + df["title"].eq("").sum()
        )

    anomalies = {
        "source": "web_source",
        "rows": len(df),
        "missing_titles": missing_titles,
    }

    print(f"Web Source : {len(df):,} lignes nettoyées")

    return df, anomalies

In [27]:
required_functions = [
    "clean_instacart",
    "clean_openfoodfacts",
    "clean_rappelconso",
    "clean_web_source",
    "clean_batch",
    "process_bronze_to_silver"
]

for name in required_functions:
    print(
        f"{name} :",
        "OK" if name in globals() else "MANQUANTE"
    )

clean_instacart : OK
clean_openfoodfacts : OK
clean_rappelconso : OK
clean_web_source : OK
clean_batch : OK
process_bronze_to_silver : OK


In [28]:
from pprint import pprint

print("=== DIAGNOSTIC SILVER ===")

paginator = s3.get_paginator("list_objects_v2")

silver_objects = []

for page in paginator.paginate(
    Bucket=RUSTFS_BUCKET,
    Prefix="silver/"
):
    for obj in page.get("Contents", []):
        silver_objects.append({
            "key": obj["Key"],
            "size_mb": round(obj["Size"] / (1024 * 1024), 2)
        })

print("\nNombre de fichiers présents dans Silver :", len(silver_objects))

pprint(silver_objects)

=== DIAGNOSTIC SILVER ===

Nombre de fichiers présents dans Silver : 6
[{'key': 'silver/instacart/aisles.parquet', 'size_mb': 0.0},
 {'key': 'silver/instacart/departments.parquet', 'size_mb': 0.0},
 {'key': 'silver/instacart/order_products__prior.parquet', 'size_mb': 117.58},
 {'key': 'silver/instacart/order_products__train.parquet', 'size_mb': 4.99},
 {'key': 'silver/instacart/orders.parquet', 'size_mb': 29.02},
 {'key': 'silver/instacart/products.parquet', 'size_mb': 1.41}]


In [29]:
transformation_results = []

for object_key in bronze_files:

    print(f"\nTraitement : {object_key}")

    result = process_bronze_to_silver(object_key)

    transformation_results.append(result)


silver_report = pd.DataFrame(
    transformation_results
)

display(silver_report)

print(
    "Nombre de fichiers envoyés dans Silver :",
    len(silver_report)
)


Traitement : bronze/instacart/aisles.parquet

Traitement : bronze/instacart/aisles.parquet
Destination : silver/instacart/aisles.parquet
Lot 1 : 134 lignes traitées
Enregistré dans RustFS : silver/instacart/aisles.parquet
Lignes : 134 => 134

Traitement : bronze/instacart/departments.parquet

Traitement : bronze/instacart/departments.parquet
Destination : silver/instacart/departments.parquet
Lot 1 : 21 lignes traitées
Enregistré dans RustFS : silver/instacart/departments.parquet
Lignes : 21 => 21

Traitement : bronze/instacart/order_products__prior.parquet

Traitement : bronze/instacart/order_products__prior.parquet
Destination : silver/instacart/order_products__prior.parquet
Lot 1 : 250,000 lignes traitées
Lot 2 : 250,000 lignes traitées
Lot 3 : 250,000 lignes traitées
Lot 4 : 250,000 lignes traitées
Lot 5 : 250,000 lignes traitées
Lot 6 : 250,000 lignes traitées
Lot 7 : 250,000 lignes traitées
Lot 8 : 250,000 lignes traitées
Lot 9 : 250,000 lignes traitées
Lot 10 : 250,000 lignes tr

ValueError: invalid literal for int() with base 10: 'openfoodfacts'

In [ ]:
run_id = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

report_csv = silver_report.to_csv(
    index=False
).encode("utf-8")

report_json = silver_report.to_json(
    orient="records",
    force_ascii=False,
    indent=2
).encode("utf-8")

csv_key = f"reports/silver/silver_quality_{run_id}.csv"
json_key = f"reports/silver/silver_quality_{run_id}.json"

s3.put_object(
    Bucket=RUSTFS_BUCKET,
    Key=csv_key,
    Body=report_csv,
    ContentType="text/csv; charset=utf-8"
)

s3.put_object(
    Bucket=RUSTFS_BUCKET,
    Key=json_key,
    Body=report_json,
    ContentType="application/json"
)

print("Rapport CSV :", csv_key)
print("Rapport JSON :", json_key)

In [ ]:
verification = []

for result in transformation_results:

    key = result["silver"]

    metadata = s3.head_object(
        Bucket=RUSTFS_BUCKET,
        Key=key
    )

    verification.append({
        "fichier_silver": key,
        "taille_octets": metadata["ContentLength"],
        "taille_attendue": result["taille_silver_octets"],
        "statut": (
            "OK"
            if metadata["ContentLength"]
            == result["taille_silver_octets"]
            else "ERREUR"
        )
    })


verification_df = pd.DataFrame(verification)

display(verification_df)

assert len(verification_df) == 9
assert (verification_df["statut"] == "OK").all()

print("VALIDATION : les 9 fichiers Silver sont présents dans RustFS.")